In [35]:
# ============================================================
# Cell 1: Setup (with pip installs)
# ============================================================
!pip install onnxruntime mir_eval librosa --quiet
!pip install basic-pitch==0.4.0 --no-deps --quiet
!pip install pretty_midi resampy --quiet
!pip install autochord tf_keras --quiet

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'    # MUST come before any autochord import
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # force CPU for Basic Pitch

import json
import numpy as np
import pandas as pd
import mir_eval
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

AUDIO_DIR       = Path('/content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles')
ANNOTATIONS_DIR = Path('/content/drive/MyDrive/Capstone/FullGuitarSetData/JamsFiles')
OUTPUT_DIR      = Path('/content/drive/MyDrive/Capstone/outputs/experiment_b')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from collections import defaultdict


In [3]:
# -------------------------
# USER CONFIG
# -------------------------

# Where outputs should go in Google Drive.
# This creates: My Drive / Capstone / outputs / fretboard_playability
# In local/non-Colab execution, this path may be created locally, but in Colab it writes to Drive after mounting.
CAPSTONE_ROOT = Path('/content/drive/MyDrive/Capstone')
OUTPUT_DIR = CAPSTONE_ROOT / 'outputs' / 'fretboard_playability'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Candidate locations for the GuitarSet data folder.
# Update/add to this list if your FullGuitarSetData or GuitarSet folder is somewhere else.
# The notebook will choose the first candidate that actually contains .jams files.
DATA_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
    Path('/content/drive/MyDrive/FullGuitarSetData'),
    Path('/content/drive/MyDrive/Capstone/GuitarSet'),
    Path('/content/drive/MyDrive/GuitarSet'),
    Path('/content/drive/MyDrive/Capstone'),
    Path('/content/drive/MyDrive'),
    Path('/mnt/data/fretwork_repo/GuitarSet'),
    Path('/mnt/data/fretwork_repo'),
]

MAX_FRET = 24
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]  # E2, A2, D3, G3, B3, E4
STRING_NAMES = ['low_E', 'A', 'D', 'G', 'B', 'high_E']
ONSET_TOLERANCE_SECONDS = 0.035

COMFORTABLE_SPAN = 5
MAX_REACHABLE_SPAN = 7
LARGE_JUMP_THRESHOLD = 5
MAX_GROUP_CANDIDATES = 25  # optimization cap for chord candidate combinations, not a demo note limit

# Tuned combined-all settings.
# `combined_all_tuned` uses empirically learned GuitarSet position priors plus adjustable weights.
# Leave RUN_WEIGHT_TUNING = True for the fastest full run. Set True if you want to run the small
# preset search below before the full evaluation.
RUN_WEIGHT_TUNING = True
TUNING_RECORD_LIMIT = 24
TUNING_OBJECTIVE_LARGE_JUMP_PENALTY = 0.35
TUNING_OBJECTIVE_DUPLICATE_STRING_PENALTY = 0.50

# Held-out evaluation settings.
# These make `combined_all_tuned` valid: train builds the position prior,
# validation selects preset weights, and test is unseen data for final reporting.
USE_HELDOUT_SPLIT = True
SPLIT_SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15


print(f'OUTPUT_DIR: {OUTPUT_DIR.resolve()}')
print('OUTPUT_DIR exists:', OUTPUT_DIR.exists())
print('\nData root candidates visible to this runtime:')
for p in DATA_ROOT_CANDIDATES:
    print(f' - {p} | exists: {p.exists()}')

if Path('/content/drive/MyDrive').exists():
    print('\nTop-level MyDrive folders/files visible to Colab:')
    for p in list(Path('/content/drive/MyDrive').iterdir())[:25]:
        print(' -', p.name)


OUTPUT_DIR: /content/drive/MyDrive/Capstone/outputs/fretboard_playability
OUTPUT_DIR exists: True

Data root candidates visible to this runtime:
 - /content/drive/MyDrive/Capstone/FullGuitarSetData | exists: True
 - /content/drive/MyDrive/FullGuitarSetData | exists: False
 - /content/drive/MyDrive/Capstone/GuitarSet | exists: True
 - /content/drive/MyDrive/GuitarSet | exists: False
 - /content/drive/MyDrive/Capstone | exists: True
 - /content/drive/MyDrive | exists: True
 - /mnt/data/fretwork_repo/GuitarSet | exists: False
 - /mnt/data/fretwork_repo | exists: False

Top-level MyDrive folders/files visible to Colab:
 - labels.csv
 - yelp_dataset.tar
 - Spring 2025
 - Tell a compelling story. Example. Fall 2020. Airline Pricing.gdoc
 - DATASCI200 Project Proposal.gdoc
 - personalized bus routes.fall 2024.gdoc
 - Final Report Template.gdoc
 - Data 201 Final Project Deliverable 2 (WORKING COPY).gdoc
 - Guitar Idea.gdoc
 - Lab 1.gdoc
 - Peer Review.gdoc
 - Lab 2 Proposal.gdoc
 - 203 Lab 3 I

In [6]:
def build_fretboard(open_string_midi=OPEN_STRING_MIDI, max_fret=MAX_FRET):
    rows = []
    for string_idx, open_midi in enumerate(open_string_midi):
        for fret in range(max_fret + 1):
            midi = open_midi + fret
            rows.append({
                'string': string_idx,
                'string_name': STRING_NAMES[string_idx],
                'fret': fret,
                'midi': midi,
                'pitch_class': midi % 12,
            })
    return pd.DataFrame(rows)

fretboard_df = build_fretboard()
MIDI_TO_POSITIONS = defaultdict(list)
for row in fretboard_df.to_dict('records'):
    MIDI_TO_POSITIONS[int(row['midi'])].append({
        'string': int(row['string']),
        'string_name': row['string_name'],
        'fret': int(row['fret']),
        'midi': int(row['midi']),
        'pitch_class': int(row['pitch_class']),
    })

def get_possible_positions(midi_note, max_fret=MAX_FRET):
    midi_note = int(round(midi_note))
    return [p for p in MIDI_TO_POSITIONS.get(midi_note, []) if 0 <= p['fret'] <= max_fret]

print('Fretboard rows:', len(fretboard_df))
display(fretboard_df.head(12))
print('Example positions for MIDI 64 / E4:')
display(pd.DataFrame(get_possible_positions(64)))


Fretboard rows: 150


,string,string_name,fret,midi,pitch_class
0,0,low_E,0,40,4
1,0,low_E,1,41,5
2,0,low_E,2,42,6
3,0,low_E,3,43,7
4,0,low_E,4,44,8
5,0,low_E,5,45,9
6,0,low_E,6,46,10
7,0,low_E,7,47,11
8,0,low_E,8,48,0
9,0,low_E,9,49,1


Example positions for MIDI 64 / E4:


,string,string_name,fret,midi,pitch_class
0,0,low_E,24,64,4
1,1,A,19,64,4
2,2,D,14,64,4
3,3,G,9,64,4
4,4,B,5,64,4
5,5,high_E,0,64,4


In [7]:
PITCH_CLASS_NAMES_SHARP = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
NOTE_TO_PC = {name: i for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}
NOTE_TO_PC.update({'Db': 1, 'Eb': 3, 'Gb': 6, 'Ab': 8, 'Bb': 10})
PC_TO_NOTE = {i: name for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}

MAJOR_STEPS = [2, 2, 1, 2, 2, 2, 1]
MINOR_STEPS = [2, 1, 2, 2, 1, 2, 2]
MAJOR_QUALITIES = ['maj', 'min', 'min', 'maj', 'maj', 'min', 'dim']
MINOR_QUALITIES = ['min', 'dim', 'maj', 'min', 'min', 'maj', 'maj']

def derive_scale(root_pc, mode='major'):
    steps = MAJOR_STEPS if mode == 'major' else MINOR_STEPS
    pcs = [root_pc]
    cur = root_pc
    for step in steps[:-1]:
        cur = (cur + step) % 12
        pcs.append(cur)
    return pcs

def build_key_database():
    rows = []
    for root_name, root_pc in NOTE_TO_PC.items():
        if 'b' in root_name:
            continue
        for mode in ['major', 'minor']:
            scale_pcs = derive_scale(root_pc, mode)
            qualities = MAJOR_QUALITIES if mode == 'major' else MINOR_QUALITIES
            chords = []
            for degree, (pc, qual) in enumerate(zip(scale_pcs, qualities), start=1):
                chords.append({
                    'degree': degree,
                    'root_pc': pc,
                    'root': PC_TO_NOTE[pc],
                    'quality': qual,
                    'symbol': f'{PC_TO_NOTE[pc]}:{qual}',
                })
            rows.append({
                'key': f'{root_name} {mode}',
                'root': root_name,
                'root_pc': root_pc,
                'mode': mode,
                'scale_pcs': scale_pcs,
                'scale_notes': [PC_TO_NOTE[pc] for pc in scale_pcs],
                'diatonic_chords': chords,
            })
    return pd.DataFrame(rows)

key_db = build_key_database()
display(key_db.head())
print('D major diatonic chords:')
d_major = key_db[key_db['key'] == 'D major'].iloc[0]
print([c['symbol'] for c in d_major['diatonic_chords']])


,key,root,root_pc,mode,scale_pcs,scale_notes,diatonic_chords
0,C major,C,0,major,"[0, 2, 4, 5, 7, 9, 11]","[C, D, E, F, G, A, B]","[{'degree': 1, 'root_pc': 0, 'root': 'C', 'qua..."
1,C minor,C,0,minor,"[0, 2, 3, 5, 7, 8, 10]","[C, D, D#, F, G, G#, A#]","[{'degree': 1, 'root_pc': 0, 'root': 'C', 'qua..."
2,C# major,C#,1,major,"[1, 3, 5, 6, 8, 10, 0]","[C#, D#, F, F#, G#, A#, C]","[{'degree': 1, 'root_pc': 1, 'root': 'C#', 'qu..."
3,C# minor,C#,1,minor,"[1, 3, 4, 6, 8, 9, 11]","[C#, D#, E, F#, G#, A, B]","[{'degree': 1, 'root_pc': 1, 'root': 'C#', 'qu..."
4,D major,D,2,major,"[2, 4, 6, 7, 9, 11, 1]","[D, E, F#, G, A, B, C#]","[{'degree': 1, 'root_pc': 2, 'root': 'D', 'qua..."


D major diatonic chords:
['D:maj', 'E:min', 'F#:min', 'G:maj', 'A:maj', 'B:min', 'C#:dim']


In [8]:
CHORD_INTERVALS = {
    'maj': [0, 4, 7],
    'min': [0, 3, 7],
    'dim': [0, 3, 6],
    'aug': [0, 4, 8],
    '7': [0, 4, 7, 10],
    'maj7': [0, 4, 7, 11],
    'min7': [0, 3, 7, 10],
    'm7': [0, 3, 7, 10],
    'sus4': [0, 5, 7],
    'sus2': [0, 2, 7],
    '5': [0, 7],
}
QUALITY_ALIASES = {'M': 'maj', 'major': 'maj', '': 'maj', 'm': 'min', 'minor': 'min', 'dom7': '7'}

def normalize_quality(q):
    if q is None:
        return 'maj'
    q = str(q).strip()
    return QUALITY_ALIASES.get(q, q)

def chord_tones(root_pc, quality='maj'):
    quality = normalize_quality(quality)
    intervals = CHORD_INTERVALS.get(quality, CHORD_INTERVALS['maj'])
    return sorted({(root_pc + i) % 12 for i in intervals})

def parse_chord_symbol(symbol):
    if symbol is None:
        return None
    s = str(symbol).strip()
    # Remove inversion/bass-note suffixes such as D:7/1 or C:maj/G before parsing quality.
    s = s.split('/')[0]
    if s in ['N', 'X', 'nan', 'None', '']:
        return None
    if ':' in s:
        root, qual = s.split(':', 1)
    else:
        m = re.match(r'^([A-G](?:#|b)?)(.*)$', s)
        if not m:
            return None
        root, qual = m.group(1), m.group(2)
    if root not in NOTE_TO_PC:
        return None
    qual = normalize_quality(qual)
    return {'root': root, 'root_pc': NOTE_TO_PC[root], 'quality': qual, 'tones': chord_tones(NOTE_TO_PC[root], qual)}

def recognize_chord_from_pitches(midi_pitches, allowed_qualities=('maj', 'min', 'dim', '7', 'maj7', 'min7')):
    pcs = sorted({int(round(m)) % 12 for m in midi_pitches})
    if not pcs:
        return None
    best = None
    for root_pc in range(12):
        for qual in allowed_qualities:
            tones = set(chord_tones(root_pc, qual))
            pcs_set = set(pcs)
            precision = len(pcs_set & tones) / max(len(pcs_set), 1)
            recall = len(pcs_set & tones) / max(len(tones), 1)
            score = 2 * precision * recall / (precision + recall + 1e-9)
            cand = {'symbol': f'{PC_TO_NOTE[root_pc]}:{qual}', 'root_pc': root_pc, 'quality': qual, 'tones': sorted(tones), 'score': score}
            if best is None or cand['score'] > best['score']:
                best = cand
    return best

print(parse_chord_symbol('D:maj'))
print(recognize_chord_from_pitches([62, 66, 69]))


{'root': 'D', 'root_pc': 2, 'quality': 'maj', 'tones': [2, 6, 9]}
{'symbol': 'D:maj', 'root_pc': 2, 'quality': 'maj', 'tones': [2, 6, 9], 'score': 0.9999999995}


In [9]:
def find_jams_dir(data_root):
    """Return a directory containing .jams files under data_root, or None if not found."""
    candidates = [
        data_root / 'JamsFiles',
        data_root / 'Annotations',
        data_root / 'GuitarSet' / 'Annotations',
        data_root / 'FullGuitarSetData' / 'JamsFiles',
        data_root / 'FullGuitarSetData' / 'Annotations',
        data_root,
    ]
    for c in candidates:
        if c.exists() and list(c.glob('*.jams')):
            return c

    # Last-resort recursive search under this candidate.
    # Limit to the first match to avoid loading the full Drive tree unnecessarily.
    if data_root.exists():
        try:
            for match in data_root.rglob('*.jams'):
                return match.parent
        except Exception as e:
            print(f'Could not recursively search {data_root}: {e}')
    return None

def choose_data_root_and_jams_dir(candidates):
    checked = []
    for root in candidates:
        checked.append((root, root.exists()))
        if not root.exists():
            continue
        jams_dir = find_jams_dir(root)
        if jams_dir is not None:
            return root, jams_dir

    print('Could not find .jams files automatically.')
    print('Checked these DATA_ROOT_CANDIDATES:')
    for root, exists in checked:
        print(f' - {root} | exists: {exists}')
    raise FileNotFoundError(
        'Could not find any .jams files. Add the correct GuitarSet/FullGuitarSetData path to DATA_ROOT_CANDIDATES.'
    )

DATA_ROOT, JAMS_DIR = choose_data_root_and_jams_dir(DATA_ROOT_CANDIDATES)
JAMS_FILES = sorted(JAMS_DIR.glob('*.jams'))
print(f'DATA_ROOT selected: {DATA_ROOT}')
print(f'Found {len(JAMS_FILES)} JAMS files in {JAMS_DIR}')
print('\n'.join(p.name for p in JAMS_FILES[:10]))

def get_annotation_data(annotation):
    data = annotation.get('data', [])
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        keys = ['time', 'duration', 'value', 'confidence']
        n = len(data.get('time', []))
        return [{k: data.get(k, [None] * n)[i] for k in keys} for i in range(n)]
    return []

def parse_string_from_data_source(data_source):
    """GuitarSet stores each string as a separate note_midi annotation with data_source 0-5."""
    try:
        s = int(data_source)
        return s if 0 <= s <= 5 else None
    except Exception:
        return None

def parse_jams_file(path):
    with open(path, 'r') as f:
        jam = json.load(f)
    notes, chords, beats = [], [], []
    tempo, key = None, None
    for ann in jam.get('annotations', []):
        ns = ann.get('namespace')
        rows = get_annotation_data(ann)
        data_source = ann.get('annotation_metadata', {}).get('data_source', '')
        if ns == 'note_midi':
            inferred_string = parse_string_from_data_source(data_source)
            for r in rows:
                v = r.get('value')
                if isinstance(v, dict):
                    midi = v.get('midi_note') or v.get('note') or v.get('pitch')
                    string = v.get('string', inferred_string)
                    fret = v.get('fret')
                else:
                    midi = v
                    string = inferred_string
                    fret = None
                if midi is None:
                    continue
                midi_int = int(round(float(midi)))
                # GuitarSet note_midi annotations usually give string via annotation data_source.
                # If fret is not explicitly stored, derive it from MIDI pitch and the open string pitch.
                if string is not None and fret is None:
                    fret = midi_int - OPEN_STRING_MIDI[int(string)]
                notes.append({
                    'start': float(r.get('time', 0.0)),
                    'duration': float(r.get('duration', 0.0) or 0.0),
                    'midi': midi_int,
                    'pitch_class': midi_int % 12,
                    'true_string': None if string is None else int(string),
                    'true_fret': None if fret is None else int(round(float(fret))),
                    'source': data_source,
                })
        elif ns in ['chord', 'chord_harte']:
            for r in rows:
                start = float(r.get('time', 0.0))
                duration = float(r.get('duration', 0.0) or 0.0)
                chord_label = r.get('value')
                chords.append({
                    'start': start,
                    'duration': duration,
                    'end': start + duration,
                    'chord': chord_label,
                    'parsed': parse_chord_symbol(chord_label),
                })
        elif ns in ['beat', 'beat_position']:
            for r in rows:
                beats.append(float(r.get('time', 0.0)))
        elif ns == 'key_mode':
            if rows:
                key = rows[0].get('value')
        elif ns == 'tempo':
            if rows:
                tempo = rows[0].get('value')
    notes = sorted(notes, key=lambda x: (x['start'], x['midi']))
    chords = sorted(chords, key=lambda x: x['start'])
    return {'recording': path.stem, 'path': str(path), 'notes': notes, 'chords': chords, 'beats': beats, 'tempo': tempo, 'key': key}

records = [parse_jams_file(p) for p in JAMS_FILES]
print('Parsed records:', len(records))
if records:
    print('Example record:', records[0]['recording'])
    print('Notes:', len(records[0]['notes']), 'Chords:', len(records[0]['chords']), 'Key:', records[0]['key'])
    display(pd.DataFrame(records[0]['notes']).head())
else:
    raise ValueError('No records parsed. Check JAMS_FILES and DATA_ROOT_CANDIDATES.')


DATA_ROOT selected: /content/drive/MyDrive/Capstone/FullGuitarSetData
Found 360 JAMS files in /content/drive/MyDrive/Capstone/FullGuitarSetData/JamsFiles
00_BN1-129-Eb_comp.jams
00_BN1-129-Eb_solo.jams
00_BN1-147-Gb_comp.jams
00_BN1-147-Gb_solo.jams
00_BN2-131-B_comp.jams
00_BN2-131-B_solo.jams
00_BN2-166-Ab_comp.jams
00_BN2-166-Ab_solo.jams
00_BN3-119-G_comp.jams
00_BN3-119-G_solo.jams
Parsed records: 360
Example record: 00_BN1-129-Eb_comp
Notes: 133 Chords: 12 Key: Eb:major


,start,duration,midi,pitch_class,true_string,true_fret,source
0,0.048816,0.423764,51,3,1,6,1
1,0.049791,0.452789,65,5,4,6,4
2,0.052717,0.458594,62,2,3,7,3
3,0.519995,0.417959,51,3,1,6,1
4,0.722036,0.859138,58,10,2,8,2


In [10]:

# -------------------------
# Valid train/validation/test split by recording
# -------------------------
# Important: split by recording, not by individual note, so notes from the same performance
# do not leak across train/validation/test.

import random


def split_records_by_recording(records, train_frac=0.70, val_frac=0.15, test_frac=0.15, seed=42):
    if not np.isclose(train_frac + val_frac + test_frac, 1.0):
        raise ValueError('train_frac + val_frac + test_frac must sum to 1.0')

    rng = random.Random(seed)

    # Keep solo/comp proportions roughly stable across splits when possible.
    groups = {
        'solo': [r for r in records if r['recording'].endswith('_solo')],
        'comp': [r for r in records if r['recording'].endswith('_comp')],
        'other': [r for r in records if not (r['recording'].endswith('_solo') or r['recording'].endswith('_comp'))],
    }

    train, val, test = [], [], []
    for label, group in groups.items():
        group = list(group)
        rng.shuffle(group)
        n = len(group)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        # Make sure split sizes add exactly to n.
        n_train = min(n_train, n)
        n_val = min(n_val, n - n_train)
        train.extend(group[:n_train])
        val.extend(group[n_train:n_train + n_val])
        test.extend(group[n_train + n_val:])

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)
    return train, val, test


TRAIN_RECORDS, VAL_RECORDS, TEST_RECORDS = split_records_by_recording(
    records,
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    seed=SPLIT_SEED,
)

print('Held-out split by recording:')
print(f'  Train records: {len(TRAIN_RECORDS)}')
print(f'  Validation records: {len(VAL_RECORDS)}')
print(f'  Test records: {len(TEST_RECORDS)}')
print(f'  Total records: {len(TRAIN_RECORDS) + len(VAL_RECORDS) + len(TEST_RECORDS)}')

split_rows = []
for split_name, split_records in [('train', TRAIN_RECORDS), ('validation', VAL_RECORDS), ('test', TEST_RECORDS)]:
    for r in split_records:
        split_rows.append({
            'recording': r['recording'],
            'split': split_name,
            'is_solo': r['recording'].endswith('_solo'),
            'is_comp': r['recording'].endswith('_comp'),
            'n_notes': len(r.get('notes', [])),
            'n_chords': len(r.get('chords', [])),
        })

split_df = pd.DataFrame(split_rows)
split_path = OUTPUT_DIR / 'fretboard_train_val_test_split.csv'
split_df.to_csv(split_path, index=False)
print('Saved split file to:', split_path.resolve())
display(split_df.groupby(['split', 'is_solo', 'is_comp']).agg(recordings=('recording', 'nunique'), notes=('n_notes', 'sum')).reset_index())


Held-out split by recording:
  Train records: 252
  Validation records: 54
  Test records: 54
  Total records: 360
Saved split file to: /content/drive/MyDrive/Capstone/outputs/fretboard_playability/fretboard_train_val_test_split.csv


,split,is_solo,is_comp,recordings,notes
0,test,False,True,27,7364
1,test,True,False,27,2843
2,train,False,True,126,31543
3,train,True,False,126,11572
4,validation,False,True,27,6708
5,validation,True,False,27,2446


In [11]:
def infer_key_from_filename(recording_name):
    parts = recording_name.split('_')
    if len(parts) >= 2:
        middle = parts[1]
        key_guess = middle.split('-')[-1]
        if key_guess in NOTE_TO_PC:
            return f'{key_guess} major'
    return None

def get_key_info(key_label):
    if key_label is None:
        return None
    s = str(key_label).replace(':', ' ').strip()
    # Normalize GuitarSet-style labels such as D:major into D major.
    toks = s.split()
    if len(toks) == 1 and toks[0] in NOTE_TO_PC:
        s = f'{toks[0]} major'
    match = key_db[key_db['key'] == s]
    return match.iloc[0].to_dict() if len(match) else None

def chord_end_time(c):
    # Some parsed chord dictionaries have duration but not an explicit end time.
    # This helper keeps the rest of the notebook robust either way.
    start = float(c.get('start', 0.0))
    if c.get('end') is not None:
        return float(c['end'])
    return start + float(c.get('duration', 0.0) or 0.0)

def chord_at_time(chords, t):
    for c in chords:
        start = float(c.get('start', 0.0))
        end = chord_end_time(c)
        if start <= t < end:
            return c
    return None

def enrich_notes_with_context(record):
    key_label = record.get('key') or infer_key_from_filename(record['recording'])
    key_info = get_key_info(key_label)
    out = []
    for n in record['notes']:
        c = chord_at_time(record['chords'], n['start'])
        row = dict(n)
        row['key_label'] = key_label
        row['in_key'] = None if key_info is None else (n['pitch_class'] in set(key_info['scale_pcs']))
        row['chord_label'] = None if c is None else c['chord']
        parsed_chord = None if c is None else c.get('parsed') or parse_chord_symbol(c.get('chord'))
        row['in_chord'] = None if parsed_chord is None else (n['pitch_class'] in set(parsed_chord['tones']))
        out.append(row)
    return out

sample_context = pd.DataFrame(enrich_notes_with_context(records[0]))
display(sample_context.head())


,start,duration,midi,pitch_class,true_string,true_fret,source,key_label,in_key,chord_label,in_chord
0,0.048816,0.423764,51,3,1,6,1,Eb:major,None,D#:maj,True
1,0.049791,0.452789,65,5,4,6,4,Eb:major,None,D#:maj,False
2,0.052717,0.458594,62,2,3,7,3,Eb:major,None,D#:maj,False
3,0.519995,0.417959,51,3,1,6,1,Eb:major,None,D#:maj,True
4,0.722036,0.859138,58,10,2,8,2,Eb:major,None,D#:maj,True


In [12]:
def estimate_hand_position_from_frets(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if not fretted else int(round(np.median(fretted)))

def group_span(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if len(fretted) <= 1 else max(fretted) - min(fretted)

def awkward_fingering_penalty(position, hand_center):
    fret = position['fret']
    if fret == 0:
        return 0.0
    distance = abs(fret - hand_center)
    if distance <= 2:
        return 0.0
    if distance <= COMFORTABLE_SPAN:
        return 0.5 * (distance - 2)
    if distance <= MAX_REACHABLE_SPAN:
        return 2.0 + (distance - COMFORTABLE_SPAN)
    return 10.0 + 2.0 * (distance - MAX_REACHABLE_SPAN)

def group_playability_cost(group_positions):
    if not group_positions:
        return 0.0
    strings = [p['string'] for p in group_positions]
    frets = [p['fret'] for p in group_positions]
    fretted = [f for f in frets if f > 0]
    cost = 0.0
    if len(strings) != len(set(strings)):
        return float('inf')
    span = group_span(frets)
    if span > COMFORTABLE_SPAN:
        cost += 2.0 * (span - COMFORTABLE_SPAN)
    if span > MAX_REACHABLE_SPAN:
        cost += 25.0 * (span - MAX_REACHABLE_SPAN)
    if fretted and min(fretted) <= 2 and max(fretted) >= 9:
        cost += 8.0
    if len(strings) >= 2:
        string_span = max(strings) - min(strings)
        if string_span > 4 and len(strings) <= 3:
            cost += 1.5 * (string_span - 4)
    hand_center = estimate_hand_position_from_frets(frets)
    cost += sum(awkward_fingering_penalty(p, hand_center) for p in group_positions)
    if any(f == 0 for f in frets) and fretted and max(fretted) > 7:
        cost += 3.0
    return cost

def transition_cost(prev_group, curr_group):
    if prev_group is None or curr_group is None:
        return 0.0
    prev_frets = [p['fret'] for p in prev_group]
    curr_frets = [p['fret'] for p in curr_group]
    prev_strings = [p['string'] for p in prev_group]
    curr_strings = [p['string'] for p in curr_group]
    prev_center = estimate_hand_position_from_frets(prev_frets)
    curr_center = estimate_hand_position_from_frets(curr_frets)
    cost = 1.2 * abs(curr_center - prev_center) + 0.25 * abs(np.mean(curr_strings) - np.mean(prev_strings))
    if len(prev_group) == 1 and len(curr_group) == 1:
        pf, cf = prev_group[0]['fret'], curr_group[0]['fret']
        ps, cs = prev_group[0]['string'], curr_group[0]['string']
        cost += 0.8 * abs(cf - pf) + 0.35 * abs(cs - ps)
        if abs(cf - pf) > LARGE_JUMP_THRESHOLD:
            cost += 4.0 + abs(cf - pf) - LARGE_JUMP_THRESHOLD
        if cf == 0 and pf > 7:
            cost += 2.0
    return cost

def context_cost(group_notes, group_positions):
    cost = 0.0
    for n, p in zip(group_notes, group_positions):
        if n.get('in_chord') is False:
            cost += 0.15
        if n.get('in_key') is False:
            cost += 0.10
    return cost


In [16]:
from itertools import product
import math


In [17]:
def group_notes_by_onset(notes, tolerance=ONSET_TOLERANCE_SECONDS):
    if not notes:
        return []
    notes_sorted = sorted(notes, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))
    groups, current = [], [notes_sorted[0]]
    group_start = notes_sorted[0]['start']
    for n in notes_sorted[1:]:
        if abs(n['start'] - group_start) <= tolerance:
            current.append(n)
        else:
            groups.append(current)
            current = [n]
            group_start = n['start']
    groups.append(current)
    return groups

def enrich_candidate(candidate):
    positions = candidate['positions']
    frets = [p['fret'] for p in positions]
    strings = [p['string'] for p in positions]
    candidate['center'] = estimate_hand_position_from_frets(frets)
    candidate['avg_string'] = float(np.mean(strings)) if strings else 0.0
    candidate['is_single'] = len(positions) == 1
    candidate['single_fret'] = positions[0]['fret'] if len(positions) == 1 else np.nan
    candidate['single_string'] = positions[0]['string'] if len(positions) == 1 else np.nan
    return candidate

def candidate_groups_for_notes(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)
    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue
        base_cost = group_playability_cost(combo) + context_cost(group_notes, combo)
        if math.isfinite(base_cost):
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))
    if not candidates:
        for combo in product(*position_lists):
            combo = list(combo)
            base_cost = group_playability_cost(combo)
            if math.isinf(base_cost):
                base_cost = 1000.0
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))
    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]

sample_groups = group_notes_by_onset(enrich_notes_with_context(records[0]))
print('Number of onset groups:', len(sample_groups))
print('First group size:', len(sample_groups[0]))
print('First group candidates:', len(candidate_groups_for_notes(sample_groups[0])))


Number of onset groups: 76
First group size: 3
First group candidates: 25


In [18]:
def choose_lowest_fret(midi):
    pos = get_possible_positions(midi)
    return None if not pos else min(pos, key=lambda p: (p['fret'], p['string']))

def choose_highest_string(midi):
    pos = get_possible_positions(midi)
    return None if not pos else max(pos, key=lambda p: (p['string'], -p['fret']))

def assign_baseline_lowest_fret(notes):
    out = []
    for n in notes:
        p = choose_lowest_fret(n['midi'])
        row = dict(n)
        row.update({'pred_string': None if p is None else p['string'], 'pred_fret': None if p is None else p['fret'], 'method': 'lowest_fret'})
        out.append(row)
    return out

def assign_baseline_highest_string(notes):
    out = []
    for n in notes:
        p = choose_highest_string(n['midi'])
        row = dict(n)
        row.update({'pred_string': None if p is None else p['string'], 'pred_fret': None if p is None else p['fret'], 'method': 'highest_string'})
        out.append(row)
    return out

def assign_nearest_previous(notes):
    groups = group_notes_by_onset(notes)
    pred_rows, prev_group = [], None
    for g in groups:
        candidates = candidate_groups_for_notes(g)
        if not candidates:
            continue
        best = min(candidates, key=lambda c: c['base_cost'] + transition_cost(prev_group, c['positions']))
        prev_group = best['positions']
        for n, p in zip(g, best['positions']):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'nearest_previous'})
            pred_rows.append(row)
    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


In [19]:
def transition_cost_matrix(prev_cands, curr_cands):
    prev_center = np.array([c['center'] for c in prev_cands], dtype=float)
    curr_center = np.array([c['center'] for c in curr_cands], dtype=float)
    prev_str = np.array([c['avg_string'] for c in prev_cands], dtype=float)
    curr_str = np.array([c['avg_string'] for c in curr_cands], dtype=float)
    mat = 1.2 * np.abs(prev_center[:, None] - curr_center[None, :])
    mat += 0.25 * np.abs(prev_str[:, None] - curr_str[None, :])

    prev_single = np.array([c['is_single'] for c in prev_cands], dtype=bool)
    curr_single = np.array([c['is_single'] for c in curr_cands], dtype=bool)
    single_mask = prev_single[:, None] & curr_single[None, :]
    if single_mask.any():
        pf = np.array([c['single_fret'] for c in prev_cands], dtype=float)[:, None]
        cf = np.array([c['single_fret'] for c in curr_cands], dtype=float)[None, :]
        ps = np.array([c['single_string'] for c in prev_cands], dtype=float)[:, None]
        cs = np.array([c['single_string'] for c in curr_cands], dtype=float)[None, :]
        fret_diff = np.abs(cf - pf)
        string_diff = np.abs(cs - ps)
        extra = 0.8 * fret_diff + 0.35 * string_diff
        extra += np.where(fret_diff > LARGE_JUMP_THRESHOLD, 4.0 + fret_diff - LARGE_JUMP_THRESHOLD, 0.0)
        extra += np.where((cf == 0) & (pf > 7), 2.0, 0.0)
        mat += np.where(single_mask, extra, 0.0)
    return mat

def assign_viterbi_playability(notes):
    groups = group_notes_by_onset(notes)
    all_candidates = [candidate_groups_for_notes(g) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = transition_cost_matrix(prev_cands, curr_cands)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'viterbi_playability'})
            pred_rows.append(row)
    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


In [20]:

# -----------------------------------------------------------------------------
# Original teammate algorithm adapted for this notebook's data structures
# -----------------------------------------------------------------------------

OLD_THEORY_WEIGHTS = {
    'key_alignment': 1.0,
    'chord_tone': 2.0,
    'open_string_bonus': 1.0,
    'low_position_bonus': 0.5,
    'middle_neck_bonus': 0.3,
    'position_continuity': 0.5,
    'continuity_cap': 5.0,
}


def old_position_score(midi, position, note_row=None, previous_position=None, weights=None):
    """Higher-is-better score from the original music-theory-aware prototype.

    This adapts the old notebook's `score_position()` logic to the richer rows in this
    notebook. The score uses key/chord flags already computed by `enrich_notes_with_context`.
    """
    if weights is None:
        weights = OLD_THEORY_WEIGHTS

    fret = position['fret']
    score = 0.0

    if note_row is not None and note_row.get('in_key') is True:
        score += weights['key_alignment']

    if note_row is not None and note_row.get('in_chord') is True:
        score += weights['chord_tone']

    if fret == 0:
        score += weights['open_string_bonus']
    elif fret <= 3:
        score += weights['low_position_bonus']
    elif 4 <= fret <= 12:
        score += weights['middle_neck_bonus']

    if previous_position is not None:
        prev_fret = previous_position['fret']
        if prev_fret > 0 and fret > 0:
            fret_distance = min(abs(fret - prev_fret), weights['continuity_cap'])
            score -= weights['position_continuity'] * (fret_distance ** 0.5)

    return float(score)


def assign_old_music_theory_greedy(notes):
    """Original teammate music-theory-aware assignment, evaluated over GuitarSet.

    Greedy per-note method:
    - enumerate valid positions for each MIDI note
    - score each position using old key/chord/comfort/continuity rules
    - choose the best local position

    For simultaneous notes, this remains per-note and can therefore reveal duplicate-string
    violations, which is useful when comparing old vs. new playability rules.
    """
    pred_rows = []
    previous_position = None

    for n in sorted(notes, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi'])):
        positions = get_possible_positions(n['midi'])
        row = dict(n)

        if not positions:
            row.update({'pred_string': None, 'pred_fret': None, 'method': 'old_music_theory_greedy'})
            pred_rows.append(row)
            continue

        best = max(
            positions,
            key=lambda p: old_position_score(n['midi'], p, note_row=n, previous_position=previous_position)
        )
        row.update({'pred_string': best['string'], 'pred_fret': best['fret'], 'method': 'old_music_theory_greedy'})
        pred_rows.append(row)
        previous_position = best

    return pred_rows


# -----------------------------------------------------------------------------
# Original/simple Viterbi without the new playability/context rules
# -----------------------------------------------------------------------------

def original_candidate_groups_for_notes(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    """Simple original-style candidate groups.

    This uses valid fretboard positions and a small low-fret preference, but does not use
    the new playability span penalties, awkward fingering penalties, open-string context,
    or chord/key context costs.
    """
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)

    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)

        # Keep physically impossible chord shapes out, but otherwise keep this simple.
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue

        frets = [p['fret'] for p in combo]
        base_cost = 0.05 * float(np.mean(frets)) + 0.05 * float(np.std(frets))
        candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))

    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]


def original_transition_cost_matrix(prev_cands, curr_cands):
    """Movement-only transition cost for the simple/original Viterbi method."""
    prev_center = np.array([c['center'] for c in prev_cands], dtype=float)
    curr_center = np.array([c['center'] for c in curr_cands], dtype=float)
    prev_str = np.array([c['avg_string'] for c in prev_cands], dtype=float)
    curr_str = np.array([c['avg_string'] for c in curr_cands], dtype=float)

    mat = np.abs(prev_center[:, None] - curr_center[None, :])
    mat += 0.25 * np.abs(prev_str[:, None] - curr_str[None, :])
    return mat


def assign_viterbi_original(notes):
    """Simple/original Viterbi assignment for comparison with new playability Viterbi."""
    groups = group_notes_by_onset(notes)
    all_candidates = [original_candidate_groups_for_notes(g) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = original_transition_cost_matrix(prev_cands, curr_cands)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'viterbi_original'})
            pred_rows.append(row)

    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


# -----------------------------------------------------------------------------
# Combined-all method: original theory score + new playability rules + Viterbi
# -----------------------------------------------------------------------------

def old_theory_group_cost(group_notes, group_positions):
    """Convert the old higher-is-better music theory score into a lower-is-better cost."""
    if not group_notes or not group_positions:
        return 0.0
    scores = []
    for n, p in zip(group_notes, group_positions):
        scores.append(old_position_score(n['midi'], p, note_row=n, previous_position=None))
    # Negative because our Viterbi minimizes cost. Scale modestly so it helps but does not dominate playability.
    return -0.35 * float(np.mean(scores))


def candidate_groups_combined_all(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    """Candidate generator combining all available signals.

    Includes:
    - valid string/fret lookup
    - duplicate-string constraint for chords
    - new playability span/stretch/open-string rules
    - new key/chord context penalties
    - old teammate music-theory score as a bonus
    """
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)

    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue

        base_cost = group_playability_cost(combo) + context_cost(group_notes, combo) + old_theory_group_cost(group_notes, combo)
        if math.isfinite(base_cost):
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))

    if not candidates:
        return candidate_groups_for_notes(group_notes, max_candidates=max_candidates)

    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]


def assign_combined_all(notes):
    """Full combined method: original theory + new playability + Viterbi sequence optimization."""
    groups = group_notes_by_onset(notes)
    all_candidates = [candidate_groups_combined_all(g) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = transition_cost_matrix(prev_cands, curr_cands)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'combined_all'})
            pred_rows.append(row)

    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


# Quick smoke test on the first record.
smoke_notes = enrich_notes_with_context(records[0])[:50]
for name, fn in {
    'old_music_theory_greedy': assign_old_music_theory_greedy,
    'viterbi_original': assign_viterbi_original,
    'combined_all': assign_combined_all,
}.items():
    smoke_pred = fn(smoke_notes)
    print(f'{name}: produced {len(smoke_pred)} predictions')


old_music_theory_greedy: produced 50 predictions
viterbi_original: produced 50 predictions
combined_all: produced 50 predictions


In [21]:

# -----------------------------------------------------------------------------
# Combined-all tuned method: empirical position priors + tuned weights + Viterbi
# -----------------------------------------------------------------------------

def build_position_prior(records, alpha=0.50):
    """Build an empirical prior over guitar positions: P(string, fret | midi).

    This is a data-driven guitaristic prior learned from GuitarSet annotations. For each
    MIDI note, it estimates how often each valid string/fret position is used in the
    annotations. It returns normalized costs where the most common position for each MIDI
    note has cost 0 and less common positions have positive cost.

    This notebook builds the prior from TRAIN_RECORDS only, then evaluates on held-out TEST_RECORDS. This avoids leakage from the test set into the learned position prior.
    """
    counts = {}
    for rec in records:
        for n in rec.get('notes', []):
            midi = n.get('midi')
            s = n.get('true_string')
            f = n.get('true_fret')
            if midi is None or s is None or f is None:
                continue
            try:
                midi = int(midi)
                s = int(s)
                f = int(f)
            except Exception:
                continue
            if not (0 <= s < len(OPEN_STRING_MIDI) and 0 <= f <= MAX_FRET):
                continue
            # Keep only physically valid ground-truth positions.
            if OPEN_STRING_MIDI[s] + f != midi:
                continue
            counts[(midi, s, f)] = counts.get((midi, s, f), 0) + 1

    prior_costs = {}
    prior_probs = {}

    for midi in range(min(MIDI_TO_POSITIONS.keys()), max(MIDI_TO_POSITIONS.keys()) + 1):
        positions = get_possible_positions(midi)
        if not positions:
            continue

        total = sum(counts.get((midi, p['string'], p['fret']), 0) for p in positions)
        denom = total + alpha * len(positions)

        raw_costs = []
        for p in positions:
            prob = (counts.get((midi, p['string'], p['fret']), 0) + alpha) / denom
            cost = -math.log(prob)
            raw_costs.append(cost)
            prior_probs[(midi, p['string'], p['fret'])] = prob

        # Normalize so the best empirical position for a MIDI note has 0 cost.
        min_cost = min(raw_costs)
        for p, cost in zip(positions, raw_costs):
            prior_costs[(midi, p['string'], p['fret'])] = cost - min_cost

    return prior_costs, prior_probs


PRIOR_SOURCE_RECORDS = TRAIN_RECORDS if USE_HELDOUT_SPLIT else records
POSITION_PRIOR_COSTS, POSITION_PRIOR_PROBS = build_position_prior(PRIOR_SOURCE_RECORDS)
print(f'Built empirical position prior from {len(PRIOR_SOURCE_RECORDS)} training records for {len(POSITION_PRIOR_COSTS)} MIDI/string/fret candidates.')


def position_prior_cost(midi, position):
    """Lower cost = position is more common for this MIDI note in GuitarSet."""
    key = (int(midi), int(position['string']), int(position['fret']))
    return float(POSITION_PRIOR_COSTS.get(key, 0.75))


DEFAULT_TUNED_WEIGHTS = {
    # Candidate/base costs
    'playability': 0.70,
    'context': 0.35,
    'old_theory': 0.45,
    'position_prior': 1.15,

    # Transition costs
    'hand_shift': 1.05,
    'string_shift': 0.22,
    'single_fret_shift': 0.65,
    'single_string_shift': 0.30,
    'large_jump_extra': 4.50,
    'open_after_high_extra': 2.25,

    # Extra group-shape preference
    'group_span_extra': 0.15,
}


def candidate_groups_combined_all_tuned(group_notes, weights=None, max_candidates=MAX_GROUP_CANDIDATES):
    """Candidate generator for the tuned combined-all method.

    It combines:
    - valid string/fret lookup
    - duplicate-string constraint for chords
    - playability rules
    - key/chord context
    - old teammate theory score
    - empirical GuitarSet position prior
    """
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS

    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)

    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)

        # Enforce physical chord feasibility.
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue

        play_cost = group_playability_cost(combo)
        if not math.isfinite(play_cost):
            continue

        ctx_cost = context_cost(group_notes, combo)
        old_cost = old_theory_group_cost(group_notes, combo)  # negative is good
        prior_cost = float(np.mean([position_prior_cost(n['midi'], p) for n, p in zip(group_notes, combo)]))

        frets = [p['fret'] for p in combo]
        span_extra = group_span(frets)

        base_cost = (
            weights['playability'] * play_cost
            + weights['context'] * ctx_cost
            + weights['old_theory'] * old_cost
            + weights['position_prior'] * prior_cost
            + weights['group_span_extra'] * span_extra
        )

        if math.isfinite(base_cost):
            cand = enrich_candidate({
                'positions': combo,
                'base_cost': float(base_cost),
                'playability_cost': float(play_cost),
                'context_cost': float(ctx_cost),
                'old_theory_cost': float(old_cost),
                'position_prior_cost': float(prior_cost),
            })
            candidates.append(cand)

    if not candidates:
        return candidate_groups_combined_all(group_notes, max_candidates=max_candidates)

    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]


def tuned_transition_cost_matrix(prev_cands, curr_cands, weights=None):
    """Transition matrix for tuned combined-all.

    Similar to the playability Viterbi transition matrix, but all major costs are
    parameterized so they can be tuned.
    """
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS

    prev_center = np.array([c['center'] for c in prev_cands], dtype=float)
    curr_center = np.array([c['center'] for c in curr_cands], dtype=float)
    prev_str = np.array([c['avg_string'] for c in prev_cands], dtype=float)
    curr_str = np.array([c['avg_string'] for c in curr_cands], dtype=float)

    mat = weights['hand_shift'] * np.abs(prev_center[:, None] - curr_center[None, :])
    mat += weights['string_shift'] * np.abs(prev_str[:, None] - curr_str[None, :])

    prev_single = np.array([c['is_single'] for c in prev_cands], dtype=bool)
    curr_single = np.array([c['is_single'] for c in curr_cands], dtype=bool)
    single_mask = prev_single[:, None] & curr_single[None, :]

    if single_mask.any():
        pf = np.array([c['single_fret'] for c in prev_cands], dtype=float)[:, None]
        cf = np.array([c['single_fret'] for c in curr_cands], dtype=float)[None, :]
        ps = np.array([c['single_string'] for c in prev_cands], dtype=float)[:, None]
        cs = np.array([c['single_string'] for c in curr_cands], dtype=float)[None, :]

        fret_diff = np.abs(cf - pf)
        string_diff = np.abs(cs - ps)
        extra = weights['single_fret_shift'] * fret_diff
        extra += weights['single_string_shift'] * string_diff
        extra += np.where(
            fret_diff > LARGE_JUMP_THRESHOLD,
            weights['large_jump_extra'] + fret_diff - LARGE_JUMP_THRESHOLD,
            0.0
        )
        extra += np.where((cf == 0) & (pf > 7), weights['open_after_high_extra'], 0.0)
        mat += np.where(single_mask, extra, 0.0)

    return mat


def assign_combined_all_tuned_with_weights(notes, weights=None, method_name='combined_all_tuned'):
    """Tuned combined-all assignment with caller-provided weights."""
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS

    groups = group_notes_by_onset(notes)
    all_candidates = [candidate_groups_combined_all_tuned(g, weights=weights) for g in groups]

    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = tuned_transition_cost_matrix(prev_cands, curr_cands, weights=weights)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': method_name})
            pred_rows.append(row)

    return sorted(
        pred_rows,
        key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi'])
    )


def assign_combined_all_tuned(notes):
    """Public method used in the full evaluation loop."""
    return assign_combined_all_tuned_with_weights(notes, weights=DEFAULT_TUNED_WEIGHTS, method_name='combined_all_tuned')


# Optional lightweight preset search. This is intentionally small so it can run in Colab.
# It updates DEFAULT_TUNED_WEIGHTS if RUN_WEIGHT_TUNING = True.
TUNED_WEIGHT_PRESETS = [
    {
        **DEFAULT_TUNED_WEIGHTS,
        'playability': 0.70,
        'old_theory': 0.45,
        'position_prior': 1.15,
        'single_fret_shift': 0.65,
    },
    {
        **DEFAULT_TUNED_WEIGHTS,
        'playability': 0.60,
        'old_theory': 0.35,
        'position_prior': 1.40,
        'single_fret_shift': 0.55,
    },
    {
        **DEFAULT_TUNED_WEIGHTS,
        'playability': 0.85,
        'old_theory': 0.30,
        'position_prior': 1.05,
        'single_fret_shift': 0.80,
    },
    {
        **DEFAULT_TUNED_WEIGHTS,
        'playability': 0.55,
        'old_theory': 0.55,
        'position_prior': 1.25,
        'single_fret_shift': 0.60,
    },
]


def tuning_objective(metrics):
    """Higher is better: accuracy with penalties for visibly bad playability."""
    return (
        float(metrics.get('exact_position_acc', 0.0))
        - TUNING_OBJECTIVE_LARGE_JUMP_PENALTY * float(metrics.get('large_jump_rate', 0.0))
        - TUNING_OBJECTIVE_DUPLICATE_STRING_PENALTY * float(metrics.get('duplicate_string_violation_rate', 0.0))
        - 0.05 * float(metrics.get('avg_fret_error', 0.0))
    )


def evaluate_weight_preset(records_subset, weights):
    rows = []
    for rec in records_subset:
        notes = enrich_notes_with_context(rec)
        notes = [
            n for n in notes
            if n.get('true_string') is not None
            and n.get('true_fret') is not None
            and 0 <= n['true_fret'] <= MAX_FRET
        ]
        if not notes:
            continue
        pred = assign_combined_all_tuned_with_weights(notes, weights=weights, method_name='combined_all_tuned_candidate')
        metrics, _ = evaluate_predictions(pred)
        rows.append(metrics)
    if not rows:
        return {'objective': -np.inf}
    df = pd.DataFrame(rows)
    agg = {
        'exact_position_acc': df['exact_position_acc'].mean(),
        'avg_fret_error': df['avg_fret_error'].mean(),
        'avg_fret_jump': df['avg_fret_jump'].mean(),
        'large_jump_rate': df['large_jump_rate'].mean(),
        'duplicate_string_violation_rate': df['duplicate_string_violation_rate'].mean(),
    }
    agg['objective'] = tuning_objective(agg)
    return agg


print('Tuned combined-all functions defined. Weight tuning will run after evaluation helpers are defined.')


Built empirical position prior from 252 training records for 150 MIDI/string/fret candidates.
Tuned combined-all functions defined. Weight tuning will run after evaluation helpers are defined.


In [25]:
def load_guitarset_recording(recording_id, audio_dir=AUDIO_DIR, annotations_dir=ANNOTATIONS_DIR):
    """
    Load a GuitarSet recording's audio path and ground truth annotations.

    Args:
        recording_id: Filename stem without extension, e.g. "00_BN1-129-Eb_comp"
        audio_dir: Path to folder containing .wav files
        annotations_dir: Path to folder containing .jams files

    Returns:
        dict with keys:
            - id: the recording_id string
            - audio_path: full path to the .wav file
            - duration: length of the recording in seconds
            - style: e.g. "BN1", "Funk1", "Jazz2"
            - tempo: BPM as int
            - key_in_filename: key as named in the filename, e.g. "Eb"
            - is_comp: True for comping, False for solo
            - key: ground truth key as labeled in the JAMS file (e.g. "Eb:major")
            - chords: list of (start, end, label) tuples for ground truth chords
            - notes: list of dicts with keys (start, duration, string, fret, midi, note_name)
            - beats: list of beat onset times in seconds
            - jam: the raw jams object (in case you need to dig deeper)

    Raises:
        FileNotFoundError: if either the audio or annotation file is missing.
    """
    audio_path = Path(audio_dir) / f"{recording_id}_mic.wav"
    jams_path = Path(annotations_dir) / f"{recording_id}.jams"

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    if not jams_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {jams_path}")

    # Parse filename for metadata
    # Format: "00_Style-TEMPO-KEY_compORsolo"
    stem = recording_id
    parts = stem.split('_')
    # parts[0] = "00", parts[1] = "BN1-129-Eb", parts[2] = "comp" or "solo"
    style_tempo_key = parts[1].split('-')
    style = style_tempo_key[0]
    tempo = int(style_tempo_key[1])
    key_in_filename = style_tempo_key[2]
    is_comp = (parts[2] == 'comp')

    # Load the JAMS annotation
    jam = jams.load(str(jams_path))

    # --- Extract key (song-level) ---
    key_label = None
    key_anns = jam.search(namespace='key_mode')
    if key_anns and len(key_anns[0].data) > 0:
        key_label = key_anns[0].data[0].value

    # --- Extract chord progression (time-aligned) ---
    chords = []
    chord_anns = jam.search(namespace='chord')
    if chord_anns:
        for obs in chord_anns[0].data:
            chords.append((obs.time, obs.time + obs.duration, obs.value))

    # --- Extract beat onsets ---
    beats = []
    beat_anns = jam.search(namespace='beat_position')
    if beat_anns:
        beats = [obs.time for obs in beat_anns[0].data]

    # --- Extract per-string note annotations and derive fret positions ---
    notes = []
    note_anns = jam.search(namespace='note_midi')
    for string_idx, anno in enumerate(note_anns):
        for obs in anno.data:
            midi_pitch = obs.value
            fret = round(midi_pitch - OPEN_STRING_MIDI[string_idx])
            notes.append({
                'start': obs.time,
                'duration': obs.duration,
                'string': string_idx,            # 0 = low E, 5 = high E
                'fret': fret,                    # 0 = open
                'midi': round(midi_pitch),
                'note_name': _midi_to_note_name(midi_pitch),
            })
    # Sort chronologically
    notes.sort(key=lambda n: n['start'])

    return {
        'id': recording_id,
        'audio_path': str(audio_path),
        'duration': jam.file_metadata.duration,
        'style': style,
        'tempo': tempo,
        'key_in_filename': key_in_filename,
        'is_comp': is_comp,
        'key': key_label,
        'chords': chords,
        'notes': notes,
        'beats': beats,
        'jam': jam,
    }


def _midi_to_note_name(midi):
    """Convert MIDI pitch number to note name string (e.g. 64 -> 'E4')."""
    pitch_classes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    octave = int(midi) // 12 - 1
    pc = pitch_classes[int(midi) % 12]
    return f"{pc}{octave}"

In [29]:
!pip install jams mir_eval autochord tf_keras onnxruntime
!pip install basic-pitch==0.4.0 --no-deps
!pip install pretty_midi resampy

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Force CPU — avoids CUDA errors with Basic Pitch

import jams
import mir_eval
import librosa
import numpy as np
import pandas as pd
import autochord
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.9 MB/s eta 0:00:00
autochord: Initializing...
autochord: Using NNLS-Chroma VAMP plugin in /root/vamp


Downloading...
From: https://drive.google.com/uc?id=1XBn7FyYjF8Ff6EuC7PjwwPzFBLRXGP7n
To: /root/.autochord/model.zip
100%|██████████| 2.26M/2.26M [00:00<00:00, 10.7MB/s]


autochord: Chord model downloaded in /root/.autochord/chroma-seq-bilstm-crf-v1/
autochord: Loaded model from /root/.autochord/chroma-seq-bilstm-crf-v1/


In [31]:
# ============================================================
# Cell 3: Pipeline Wrappers
# ============================================================
# Thin wrappers around our three detection tools.
# Each function takes an audio path and returns predictions
# in a format that matches the corresponding ground truth field
# from load_guitarset_recording().

from basic_pitch.inference import predict as basic_pitch_predict


# ---------- Key Detection ----------

# Krumhansl-Schmuckler key profiles
_MAJOR_PROFILE = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09,
                            2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
_MINOR_PROFILE = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53,
                            2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
_PITCH_CLASSES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']


def run_key_detection(audio_path):
    """
    Detect the key of an audio file using Krumhansl-Schmuckler on chromagram.

    Returns:
        dict with keys:
            - key: predicted key label, e.g. "D# major"
            - confidence_gap: gap between top-1 and top-2 (rough confidence)
            - ranked: top 5 candidates with correlation scores
    """
    y, sr = librosa.load(audio_path, sr=None)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    chroma_avg = np.mean(chroma, axis=1)

    correlations = {}
    for tonic in range(12):
        major_template = np.roll(_MAJOR_PROFILE, tonic)
        minor_template = np.roll(_MINOR_PROFILE, tonic)
        correlations[f"{_PITCH_CLASSES[tonic]} major"] = np.corrcoef(chroma_avg, major_template)[0, 1]
        correlations[f"{_PITCH_CLASSES[tonic]} minor"] = np.corrcoef(chroma_avg, minor_template)[0, 1]

    ranked = sorted(correlations.items(), key=lambda x: x[1], reverse=True)
    top_key, top_score = ranked[0]
    _, second_score = ranked[1]

    return {
        'key': top_key,
        'confidence_gap': top_score - second_score,
        'ranked': ranked[:5],
    }


# ---------- Chord Detection ----------

def run_chord_detection(audio_path):
    """
    Detect chord progression using autochord.

    Returns:
        list of (start, end, label) tuples in the same shape as
        ground truth chord annotations from the loader.
    """
    chords = autochord.recognize(audio_path)
    # autochord already returns (start, end, label) tuples
    return [(start, end, label) for start, end, label in chords]


# ---------- Note Detection ----------

def run_note_detection(audio_path):
    """
    Detect notes using Basic Pitch.

    Returns:
        list of dicts with keys (start, duration, midi, note_name).
        Note: no string/fret info — that's our fretboard algorithm's job later.
    """
    _, _, note_events = basic_pitch_predict(audio_path)
    notes = []
    for start, end, pitch_midi, amplitude, _ in note_events:
        # Filter out low-amplitude artifacts and out-of-guitar-range notes
        if amplitude < 0.3 or pitch_midi < 40 or pitch_midi > 88:
            continue
        notes.append({
            'start': start,
            'duration': end - start,
            'midi': int(round(pitch_midi)),
            'note_name': _midi_to_note_name(pitch_midi),
            'amplitude': amplitude,
        })
    notes.sort(key=lambda n: n['start'])
    return notes


test_id = "00_BN1-129-Eb_comp"
rec = parse_jams_file(JAMS_DIR / f"{test_id}.jams")
audio_path = str(AUDIO_DIR / f"{test_id}_mic.wav")

print(f"Recording: {test_id}")
print(f"GT notes: {len(rec['notes'])}")
print()
print("Running detection pipeline...")
notes = run_note_detection(audio_path)
chords = run_chord_detection(audio_path)
key = run_key_detection(audio_path)
print(f"  Detected notes:  {len(notes)}")
print(f"  Chord segments:  {len(chords)}")
print(f"  Detected key:    {key.get('key') if isinstance(key, dict) else key}")

Recording: 00_BN1-129-Eb_comp
GT notes: 133

Running detection pipeline...
Predicting MIDI for /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_BN1-129-Eb_comp_mic.wav...
1/1 [==============================] - 0s 52ms/step
  Detected notes:  147
  Chord segments:  9
  Detected key:    D# major


In [32]:
# ============================================================
# Cell 12: Note matching (NEW)
# ============================================================
def match_detected_to_truth(detected_notes, gt_notes,
                            onset_tolerance=0.05, pitch_tolerance=50):
    """Match Basic Pitch's detected notes to GuitarSet ground truth notes.

    Uses mir_eval's standard transcription matching: a detected note matches
    a GT note if their onsets are within `onset_tolerance` seconds and their
    pitches are within `pitch_tolerance` cents. Each GT note can match at
    most one detected note and vice versa.

    Args:
        detected_notes: list of dicts with 'start', 'duration', 'midi'
        gt_notes:       list of dicts with 'start', 'duration', 'midi'
        onset_tolerance: seconds (default 0.05 = 50ms, mir_eval standard)
        pitch_tolerance: cents (default 50 = a quarter-tone)

    Returns:
        list of (gt_idx, det_idx) tuples — indices into gt_notes and
        detected_notes for each matched pair.
    """
    if not gt_notes or not detected_notes:
        return []

    gt_intervals  = np.array([[n['start'], n['start'] + n['duration']]
                              for n in gt_notes])
    gt_pitches    = np.array([float(n['midi']) for n in gt_notes])
    det_intervals = np.array([[n['start'], n['start'] + n['duration']]
                              for n in detected_notes])
    det_pitches   = np.array([float(n['midi']) for n in detected_notes])

    matches = mir_eval.transcription.match_notes(
        gt_intervals, gt_pitches,
        det_intervals, det_pitches,
        onset_tolerance=onset_tolerance,
        pitch_tolerance=pitch_tolerance,
    )
    # mir_eval returns (ref_idx, est_idx) pairs as a list
    return list(matches)

In [39]:
# ============================================================
# Cell 13: End-to-end evaluation for a single recording (NEW)
# ============================================================
def evaluate_recording_end_to_end(recording_id, verbose=False):
    """Run the full audio → tab pipeline on one recording and score
    against ground truth using matched-note position accuracy."""
    audio_path = AUDIO_DIR / f"{recording_id}_mic.wav"
    jams_path  = JAMS_DIR / f"{recording_id}.jams"

    gt_record = parse_jams_file(jams_path)
    gt_notes = [n for n in gt_record['notes']
                if n['true_string'] is not None
                and n['true_fret'] is not None]
    if not gt_notes:
        return None

    # Detection pipeline
    detected_notes  = run_note_detection(str(audio_path))
    detected_chords_raw = run_chord_detection(str(audio_path))
    detected_key_raw    = run_key_detection(str(audio_path))

    # Convert eval_pipeline's tuple format into the dict format
    # the teammate's algorithm expects.
    detected_chords = [
        {
            'start':    float(s),
            'duration': float(e - s),
            'end':      float(e),
            'chord':    label,
            'parsed':   parse_chord_symbol(label) if label else None,
        }
        for s, e, label in detected_chords_raw
    ]

    # Key: eval_pipeline returns a dict, teammate expects a string label
    if isinstance(detected_key_raw, dict):
        key_label = detected_key_raw.get('key')
    else:
        key_label = detected_key_raw

    # Enrich detected notes with the fields the teammate's algorithm expects
    # (pitch_class, plus placeholder true_string/true_fret since we don't
    # have ground truth at inference time).
    detected_notes_enriched = [
        {
            'start':       float(n['start']),
            'duration':    float(n.get('duration', 0.0)),
            'midi':        int(n['midi']),
            'pitch_class': int(n['midi']) % 12,
            'true_string': None,
            'true_fret':   None,
            'source':      'basic_pitch',
        }
        for n in detected_notes
    ]

    fake_record = {
        'recording': recording_id,
        'notes':     detected_notes_enriched,   # ← use the enriched list
        'chords':    detected_chords,
        'key':       key_label,
        'beats':     [],
        'tempo':     None,
    }
    enriched = enrich_notes_with_context(fake_record)
    predictions = assign_combined_all_tuned(enriched)

    matches = match_detected_to_truth(detected_notes_enriched, gt_notes)

    n_gt        = len(gt_notes)
    n_detected  = len(detected_notes)
    n_matched   = len(matches)

    correct_position = 0
    for gt_idx, det_idx in matches:
        gt_n  = gt_notes[gt_idx]
        pred  = predictions[det_idx]
        if (pred.get('pred_string') == gt_n['true_string'] and
            pred.get('pred_fret')   == gt_n['true_fret']):
            correct_position += 1

    detection_rate     = n_matched / n_gt if n_gt else 0.0
    assignment_on_hits = correct_position / n_matched if n_matched else 0.0
    joint_rate         = correct_position / n_gt if n_gt else 0.0

    result = {
        'recording':           recording_id,
        'n_gt':                n_gt,
        'n_detected':          n_detected,
        'n_matched':           n_matched,
        'n_correct':           correct_position,
        'detection_rate':      detection_rate,
        'assignment_on_hits':  assignment_on_hits,
        'joint_rate':          joint_rate,
    }
    if verbose:
        print(f"{recording_id}")
        print(f"  GT notes: {n_gt}   Detected: {n_detected}   Matched: {n_matched}")
        print(f"  Detection rate (matched / GT): {detection_rate:.3f}")
        print(f"  Assignment on matched:         {assignment_on_hits:.3f}")
        print(f"  Joint end-to-end rate:         {joint_rate:.3f}")
    return result

In [40]:
# ============================================================
# Cell 14: Sanity check on one recording (NEW)
# ============================================================
# Pick a recording in your held-out test split. If you have TEST_RECORDS
# from cell 10, grab the first one. Otherwise hardcode any test recording.
test_rid = TEST_RECORDS[0]['recording']
result = evaluate_recording_end_to_end(test_rid, verbose=True)

Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/02_BN2-131-B_solo_mic.wav...
1/1 [==============================] - 0s 67ms/step
02_BN2-131-B_solo
  GT notes: 66   Detected: 83   Matched: 48
  Detection rate (matched / GT): 0.727
  Assignment on matched:         0.562
  Joint end-to-end rate:         0.409


In [41]:
# ============================================================
# Cell 15: Batch evaluation on test set (NEW)
# ============================================================
results = []
for i, rec in enumerate(TEST_RECORDS):
    rid = rec['recording']
    print(f"[{i+1}/{len(TEST_RECORDS)}] {rid}", end='  ')
    try:
        r = evaluate_recording_end_to_end(rid)
        if r is None:
            print("(skipped: no GT notes)")
            continue
        results.append(r)
        print(f"joint={r['joint_rate']:.3f}")
    except Exception as e:
        print(f"FAILED: {e}")

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / 'experiment_b_per_recording.csv', index=False)
print(f"\nSaved {len(results_df)} recordings to "
      f"{OUTPUT_DIR / 'experiment_b_per_recording.csv'}")

[1/54] 02_BN2-131-B_solo  Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/02_BN2-131-B_solo_mic.wav...
1/1 [==============================] - 0s 80ms/step
joint=0.409
[2/54] 00_Rock2-142-D_comp  Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_Rock2-142-D_comp_mic.wav...
1/1 [==============================] - 0s 107ms/step
joint=0.121
[3/54] 05_Rock2-85-F_solo  Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/05_Rock2-85-F_solo_mic.wav...
1/1 [==============================] - 0s 54ms/step
joint=0.664
[4/54] 04_Jazz3-137-Eb_comp  Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/04_Jazz3-137-Eb_comp_mic.wav...
1/1 [==============================] - 0s 50ms/step
joint=0.114
[5/54] 05_SS2-88-F_solo  Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/05_SS2-88-F_solo_mic.wav...
1/1 [==============================] - 0s 87ms/step
j

In [42]:
# ============================================================
# Cell 16: Summary comparison — Experiment A vs B (NEW)
# ============================================================
total_gt        = results_df['n_gt'].sum()
total_detected  = results_df['n_detected'].sum()
total_matched   = results_df['n_matched'].sum()
total_correct   = results_df['n_correct'].sum()

agg_detection  = total_matched / total_gt
agg_assignment = total_correct / total_matched if total_matched else 0.0
agg_joint      = total_correct / total_gt

print("=" * 60)
print(f"EXPERIMENT B — END-TO-END ON HELD-OUT TEST")
print(f"({len(results_df)} recordings, {total_gt} GT notes)")
print("=" * 60)
print(f"Detection rate (matched / GT):         {agg_detection:.3f}")
print(f"Assignment accuracy on matched notes:  {agg_assignment:.3f}")
print(f"Joint end-to-end rate:                 {agg_joint:.3f}")
print()
print("Comparison to Experiment A (GT MIDI input):")
print(f"  Experiment A position accuracy:      0.688")
print(f"  Experiment B assignment on matched:  {agg_assignment:.3f}")
print(f"  Cost of detection error:             "
      f"{0.688 - agg_assignment:+.3f}")
print()
print("End-to-end joint rate is the number a user actually experiences:")
print(f"  Fraction of original notes fully correct: {agg_joint:.1%}")

EXPERIMENT B — END-TO-END ON HELD-OUT TEST
(54 recordings, 10207 GT notes)
Detection rate (matched / GT):         0.506
Assignment accuracy on matched notes:  0.386
Joint end-to-end rate:                 0.196

Comparison to Experiment A (GT MIDI input):
  Experiment A position accuracy:      0.688
  Experiment B assignment on matched:  0.386
  Cost of detection error:             +0.302

End-to-end joint rate is the number a user actually experiences:
  Fraction of original notes fully correct: 19.6%
